# Notebook 5: Daten Laden - L in ETL

## Lernziele

Nach diesem Notebook kannst du:
- DataFrames als CSV und Excel exportieren
- Daten in eine SQLite-Datenbank schreiben und lesen
- JSON-Dateien schreiben
- Einfache SQL-Abfragen mit pandas ausführen
- Eine vollständige ETL-Pipeline als Funktion zusammenbauen

---

## Der Load-Schritt

Am Ende einer ETL-Pipeline stehen die bereinigten Daten. Sie müssen an einen **Zielort** geschrieben werden:

```
Zielorte:
  - Dateien: CSV, Excel, JSON, Parquet
  - Datenbanken: SQLite, PostgreSQL, MySQL, SQL Server
  - Cloud: Data Warehouses (Snowflake, BigQuery, Redshift)
```

Wir fokussieren uns auf CSV/Excel (universell) und SQLite (einfachste Datenbank, kein Server nötig).

In [ ]:
import pandas as pd
import sqlite3
import os

os.makedirs("output", exist_ok=True)
os.makedirs("daten", exist_ok=True)
print("Bereit")

In [ ]:
# Beispiel-DataFrame für dieses Notebook
df = pd.DataFrame([
    {"id": 1, "produkt": "Laptop Pro",  "kategorie": "Elektronik", "preis": 999.99, "menge": 2, "datum": "2024-01-15", "region": "nord"},
    {"id": 2, "produkt": "USB-Maus",    "kategorie": "Zubehör",    "preis": 29.90,  "menge": 5, "datum": "2024-01-15", "region": "süd"},
    {"id": 3, "produkt": "Monitor",     "kategorie": "Elektronik", "preis": 349.00, "menge": 1, "datum": "2024-01-16", "region": "ost"},
    {"id": 4, "produkt": "Tastatur",    "kategorie": "Zubehör",    "preis": 49.90,  "menge": 3, "datum": "2024-01-16", "region": "nord"},
    {"id": 5, "produkt": "Webcam HD",   "kategorie": "Zubehör",    "preis": 79.00,  "menge": 2, "datum": "2024-01-17", "region": "süd"},
    {"id": 6, "produkt": "SSD 1TB",     "kategorie": "Speicher",   "preis": 89.90,  "menge": 4, "datum": "2024-01-17", "region": "ost"},
    {"id": 7, "produkt": "RAM 16GB",    "kategorie": "Speicher",   "preis": 65.00,  "menge": 3, "datum": "2024-01-18", "region": "nord"},
    {"id": 8, "produkt": "Headset Pro", "kategorie": "Zubehör",    "preis": 89.00,  "menge": 1, "datum": "2024-01-18", "region": "süd"},
])

df["datum"] = pd.to_datetime(df["datum"])
df["umsatz"] = df["preis"] * df["menge"]
df

---
# 1. In CSV exportieren

In [ ]:
# Standard-Export (englisches Format, Komma als Trennzeichen)
df.to_csv("output/verkauf_export.csv", index=False)  # index=False: keinen Zeilenindex mitschreiben!
print("Gespeichert: output/verkauf_export.csv")

# Deutsches Format (Semikolon, Komma als Dezimaltrenner)
df.to_csv(
    "output/verkauf_export_de.csv",
    sep=";",
    decimal=",",
    encoding="utf-8-sig",  # utf-8-sig: Excel öffnet Umlaute korrekt
    index=False
)
print("Gespeichert: output/verkauf_export_de.csv")

# Datei wieder einlesen und prüfen
check = pd.read_csv("output/verkauf_export.csv")
print(f"\nZeilen eingelesen: {len(check)}, Spalten: {list(check.columns)}")

### Nützliche `to_csv()` Parameter

In [ ]:
# Nur bestimmte Spalten exportieren
df[["id", "produkt", "umsatz"]].to_csv("output/umsatz_kurz.csv", index=False)

# Anzahl Nachkommastellen begrenzen
df.to_csv("output/verkauf_gerundet.csv", index=False, float_format="%.2f")

print("Alle CSV-Exporte abgeschlossen.")
print("Dateien in output/:", os.listdir("output"))

---
# 2. In Excel exportieren

In [ ]:
# Einfacher Export
df.to_excel("output/verkauf.xlsx", sheet_name="Verkauf", index=False)
print("Gespeichert: output/verkauf.xlsx")

# Mehrere Sheets in einer Datei
# Aggregation: Umsatz pro Kategorie
df_kat = df.groupby("kategorie").agg(
    umsatz_gesamt=("umsatz", "sum"),
    anzahl_bestellungen=("id", "count")
).reset_index()

# Aggregation: Umsatz pro Region
df_reg = df.groupby("region")["umsatz"].sum().reset_index()

with pd.ExcelWriter("output/bericht.xlsx", engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Rohdaten", index=False)
    df_kat.to_excel(writer, sheet_name="Nach Kategorie", index=False)
    df_reg.to_excel(writer, sheet_name="Nach Region", index=False)

print("Gespeichert: output/bericht.xlsx (3 Sheets)")

---
# 3. In JSON exportieren

In [ ]:
# Als JSON-Array (Liste von Objekten - am gebräuchlichsten)
df_json_export = df.copy()
df_json_export["datum"] = df_json_export["datum"].dt.strftime("%Y-%m-%d")  # Datum als String

df_json_export.to_json(
    "output/verkauf.json",
    orient="records",   # Format: [{...}, {...}, ...]
    indent=2,           # Einrückung für Lesbarkeit
    force_ascii=False   # Umlaute nicht escapen
)

# Datei anzeigen
with open("output/verkauf.json", encoding="utf-8") as f:
    inhalt = f.read()
print(inhalt[:500], "...")

---
# 4. SQLite - Daten in eine Datenbank laden

**SQLite** ist eine eingebettete Datenbank - sie braucht keinen Server und ist ideal zum Lernen und für kleinere Projekte.

SQL (Structured Query Language) ist die Standardsprache für relationale Datenbanken.

In [ ]:
# Datenbankdatei erstellen (oder öffnen)
db_pfad = "output/verkauf.db"
verbindung = sqlite3.connect(db_pfad)
print(f"Datenbankverbindung geöffnet: {db_pfad}")

# DataFrame in Tabelle schreiben
df_fuer_db = df.copy()
df_fuer_db["datum"] = df_fuer_db["datum"].dt.strftime("%Y-%m-%d")  # SQLite mag ISO-Format

df_fuer_db.to_sql(
    name="verkauf",           # Tabellenname
    con=verbindung,           # Datenbankverbindung
    if_exists="replace",      # 'replace' überschreibt, 'append' fügt hinzu, 'fail' wirft Fehler
    index=False               # Keinen pandas-Index als Spalte speichern
)
print(f"Tabelle 'verkauf' geschrieben: {len(df_fuer_db)} Zeilen")

### SQL-Abfragen — SELECT Grundlagen

SQL liest sich fast wie natürliche Sprache. Mehrzeilige Strings (`"""`) eignen sich gut für SQL-Abfragen — sie erhalten Zeilenumbrüche und Einrückungen.

In [ ]:
# Daten aus der Datenbank lesen
df_aus_db = pd.read_sql("SELECT * FROM verkauf", verbindung)
print(f"Aus DB gelesen: {df_aus_db.shape}")
df_aus_db.head()

In [ ]:
# Alle Elektronik-Produkte
sql_1 = """
    SELECT produkt, preis, menge, umsatz
    FROM verkauf
    WHERE kategorie = 'Elektronik'
    ORDER BY umsatz DESC
"""
print("Elektronik-Produkte:")
display(pd.read_sql(sql_1, verbindung))

In [ ]:
# GROUP BY - Aggregation in SQL
sql_2 = """
    SELECT
        kategorie,
        COUNT(*) AS anzahl,
        SUM(umsatz) AS umsatz_gesamt,
        ROUND(AVG(preis), 2) AS preis_durchschnitt
    FROM verkauf
    GROUP BY kategorie
    ORDER BY umsatz_gesamt DESC
"""
print("Umsatz pro Kategorie:")
display(pd.read_sql(sql_2, verbindung))

In [ ]:
# Zweite Tabelle: Produktkatalog
produkt_katalog = pd.DataFrame([
    {"produkt": "Laptop Pro",  "hersteller": "TechCorp",  "garantie_monate": 24},
    {"produkt": "USB-Maus",    "hersteller": "AccessIT",   "garantie_monate": 12},
    {"produkt": "Monitor",     "hersteller": "TechCorp",   "garantie_monate": 36},
    {"produkt": "Tastatur",    "hersteller": "AccessIT",   "garantie_monate": 12},
    {"produkt": "Webcam HD",   "hersteller": "ViewTech",   "garantie_monate": 12},
])

produkt_katalog.to_sql("katalog", verbindung, if_exists="replace", index=False)

# JOIN in SQL - Tabellen verknüpfen
sql_3 = """
    SELECT
        v.produkt,
        v.umsatz,
        k.hersteller,
        k.garantie_monate
    FROM verkauf AS v
    LEFT JOIN katalog AS k ON v.produkt = k.produkt
    ORDER BY v.umsatz DESC
"""
print("Verkauf + Katalog (JOIN):")
display(pd.read_sql(sql_3, verbindung))

In [ ]:
# Verbindung schließen (wichtig!)
verbindung.close()
print("Datenbankverbindung geschlossen.")

---
### Übung 1: Daten exportieren

1. Erstelle eine Aggregationstabelle `df_zusammenfassung` mit Umsatz und Anzahl Bestellungen pro Region und Kategorie.
2. Exportiere diese Tabelle als CSV nach `output/zusammenfassung.csv`.
3. Schreibe den Original-DataFrame `df` und die Zusammenfassung in eine SQLite-Datenbank `output/analyse.db`.
4. Lese die Zusammenfassung per SQL-Abfrage zurück und zeige nur Regionen mit Gesamtumsatz > 200 Euro.

In [ ]:
# Deine Lösung:


---
# 5. Die vollständige ETL-Pipeline

Jetzt bringen wir alles zusammen: Extract, Transform, Load als eine wiederverwendbare Pipeline.

In [ ]:
# Testdatei erstellen (simuliert eine tägliche Lieferdatei)
testdaten = """id,produkt,kategorie,preis,menge,datum,region
101,Laptop Pro,Elektronik,999.99,2,2024-02-01,nord
102,USB-Maus,Zubehör,29.90,5,2024-02-01,süd
103,Monitor,Elektronik,,1,2024-02-02,ost
104,Tastatur,Zubehör,49.90,3,2024-02-02,nord
105,SSD 1TB,Speicher,89.90,4,2024-02-03,süd
"""
with open("daten/tagesdaten.csv", "w") as f:
    f.write(testdaten)

In [ ]:
# === VOLLSTÄNDIGE ETL-PIPELINE ===

def extract(dateipfad):
    """E: Daten aus CSV-Datei lesen."""
    print(f"[EXTRACT] Lese: {dateipfad}")
    df = pd.read_csv(dateipfad, parse_dates=["datum"])
    print(f"  -> {len(df)} Zeilen eingelesen")
    return df


def transform(df):
    """T: Daten bereinigen und anreichern."""
    print(f"[TRANSFORM] Verarbeite {len(df)} Zeilen...")
    df = df.copy()

    # Fehlende Preise mit Kategorie-Durchschnitt füllen
    preis_mittel = df.groupby("kategorie")["preis"].transform("mean")
    n_fehlend = df["preis"].isnull().sum()
    df["preis"] = df["preis"].fillna(preis_mittel)
    if n_fehlend > 0:
        print(f"  Fehlende Preise ergänzt: {n_fehlend}")

    # Umsatz berechnen
    df["umsatz_netto"] = df["preis"] * df["menge"]
    df["umsatz_brutto"] = (df["umsatz_netto"] * 1.19).round(2)

    # Texte bereinigen
    df["region"] = df["region"].str.lower()
    df["region"] = df["region"].str.strip()
    df["kategorie"] = df["kategorie"].str.strip()

    # Zeitdimension
    df["woche"] = df["datum"].dt.isocalendar().week
    df["woche"] = df["woche"].astype(int)
    df["monat"] = df["datum"].dt.month

    print(f"  -> {len(df)} Zeilen transformiert")
    return df


def load(df, db_pfad, tabellenname, modus="append"):
    """L: Daten in SQLite-Datenbank schreiben."""
    print(f"[LOAD] Schreibe in {db_pfad} (Tabelle: {tabellenname}, Modus: {modus})")
    
    df_export = df.copy()
    df_export["datum"] = df_export["datum"].dt.strftime("%Y-%m-%d")
    
    con = sqlite3.connect(db_pfad)
    df_export.to_sql(tabellenname, con, if_exists=modus, index=False)
    
    zeilen_df = pd.read_sql(f"SELECT COUNT(*) as n FROM {tabellenname}", con)
    zeilen = zeilen_df.iloc[0, 0]
    con.close()
    
    print(f"  -> {len(df)} Zeilen geladen. Tabelle gesamt: {zeilen} Zeilen")


def etl_pipeline(quelldatei, ziel_db, tabellenname):
    """Führt die komplette ETL-Pipeline aus."""
    print("=" * 50)
    print(f"ETL-Pipeline gestartet")
    print(f"  Quelle: {quelldatei}")
    print(f"  Ziel:   {ziel_db}/{tabellenname}")
    print("=" * 50)
    
    df_roh = extract(quelldatei)
    df_fertig = transform(df_roh)
    load(df_fertig, ziel_db, tabellenname)
    print("\nPipeline erfolgreich abgeschlossen!")
    return df_fertig


# Pipeline ausführen!
df_ergebnis = etl_pipeline(
    quelldatei="daten/tagesdaten.csv",
    ziel_db="output/pipeline.db",
    tabellenname="verkauf"
)

In [ ]:
# Ergebnis anzeigen
df_ergebnis

In [ ]:
# Datenbank-Ergebnis abfragen
con = sqlite3.connect("output/pipeline.db")
ergebnis = pd.read_sql("""
    SELECT
        kategorie,
        COUNT(*) AS bestellungen,
        ROUND(SUM(umsatz_netto), 2) AS netto,
        ROUND(SUM(umsatz_brutto), 2) AS brutto
    FROM verkauf
    GROUP BY kategorie
    ORDER BY brutto DESC
""", con)
con.close()

print("Ergebnis aus der Datenbank:")
ergebnis

---
### Übung 2: Pipeline erweitern

Erweitere die `transform()`-Funktion um einen zusätzlichen Schritt:

Eine neue Spalte `wert_kategorie`:
- `"hoch"` wenn `umsatz_netto` > 500
- `"mittel"` wenn `umsatz_netto` zwischen 100 und 500
- `"niedrig"` wenn `umsatz_netto` < 100

Führe die Pipeline danach erneut aus und zeige die Verteilung der Wert-Kategorien.

In [ ]:
# Deine Lösung:


---
## Zusammenfassung

In diesem Notebook hast du gelernt, transformierte Daten zu speichern:

| Zielformat | Funktion / Methode |
|------------|-------------------|
| **CSV** | `to_csv()` - Standard- und deutsches Format |
| **Excel** | `to_excel()`, mehrere Sheets mit `ExcelWriter` |
| **JSON** | `to_json()` mit verschiedenen `orient`-Optionen |
| **SQLite** | `to_sql()`, SQL-Abfragen mit `SELECT`, `WHERE`, `GROUP BY`, `JOIN` |

Als Abschluss hast du eine vollständige **ETL-Pipeline** gebaut: `extract()` → `transform()` → `load()`.

---
# Weitere Übungsaufgaben

Die folgenden Aufgaben beziehen sich auf die Themen dieses Notebooks. Musterlösungen findest du im Ordner `Loesungen/`.

### Aufgabe 1 - Gefilterter Export

Exportiere aus dem Bestellungs-DataFrame:
1. Alle Bestellungen → `output/alle_bestellungen.csv`
2. Nur offene Bestellungen → `output/offene_bestellungen.csv`
3. Umsatz-Zusammenfassung pro Status → `output/zusammenfassung.csv`

Lese jede Datei danach wieder ein und verifiziere die Zeilenanzahl.

In [ ]:
import pandas as pd, os
os.makedirs('output', exist_ok=True)

bestellungen = pd.DataFrame({
    'id':     range(1, 11),
    'kunde':  ['Anna','Klaus','Sara','Tom','Lisa','Marc','Jana','Paul','Eva','Ben'],
    'betrag': [120, 450, 89, 1200, 34, 780, 220, 95, 540, 180],
    'status': ['geliefert','offen','geliefert','offen','storniert',
               'geliefert','offen','geliefert','offen','geliefert'],
})

# Deine Lösung:


### Aufgabe 2 - SQLite Abfragen

Schreibe den Bestellungs-DataFrame in eine SQLite-Datenbank.
Beantworte folgende Fragen **ausschließlich mit SQL**:
1. Wie viele Bestellungen hat jeder Status?
2. Welcher Kunde hat den höchsten Gesamtbetrag?
3. Wie hoch ist der Anteil offener Bestellungen am Gesamtumsatz (in %)?

In [ ]:
import pandas as pd, sqlite3

# bestellungen-DataFrame von Aufgabe 1 vorausgesetzt
# Deine Lösung:
